# Глава 10. CatBoost и LightGBM на своём компьютере

Практика к урокам «CatBoost и LightGBM в своём Jupyter» и «Проект: прогноз дневной выручки». В браузере курса этих библиотек нет: это скомпилированный код на C++, его не собирают для Python в браузере. Поэтому работаем в своём Jupyter.

**Что сделаем**

1. Возьмём данные и признаки проекта главы 10: `daily_sales.csv`, выручка шести кофеен по дням.
2. Разделим данные по времени на три части: обучение, валидация для ранней остановки и тест.
3. Обучим бустинги с ранней остановкой: `HistGradientBoostingRegressor` из sklearn с флагами `get_dummies` и с нативными категориями, LightGBM в двух настройках и CatBoost.
4. Соберём таблицу «модель | RMSE | MAE | смещение | деревьев | fit, с | predict, мс».

**Перед запуском**

- Установите библиотеки в окружение курса (глава 2): `uv add catboost lightgbm` или `pip install catboost lightgbm`.
- Блокнот сам ищет `daily_sales.csv`: в `../../data` (папка курса), `../data`, `data` или рядом с собой.
- Все библиотеки работают на одном потоке процессора (`THREADS = 1` в первой ячейке). Так сравнение времени честнее: каждая библиотека по-своему использует ядра, и на загруженном компьютере многопоточность может даже замедлить обучение. Хотите быстрее — поставьте `THREADS` равным числу ядер и перезапустите блокнот.
- Сохранённые выводы получены на нашем компьютере. Секунды у вас будут другими, а качество (RMSE, MAE) должно совпасть с точностью до версий библиотек.

In [ ]:
import os
THREADS = 1                                     # все библиотеки на одном потоке, см. пояснение выше
os.environ["OMP_NUM_THREADS"] = str(THREADS)    # задаём до импорта sklearn и lightgbm

import time
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import lightgbm as lgb
import catboost
from catboost import CatBoostRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

RANDOM_STATE = 0
DATA = next(p for p in [Path("../../data"), Path("../data"), Path("data"), Path(".")]
            if (p / "daily_sales.csv").exists())

print("pandas", pd.__version__, "| sklearn", sklearn.__version__,
      "| lightgbm", lgb.__version__, "| catboost", catboost.__version__)
print("данные:", DATA.resolve())

## 1. Данные и признаки

Признаки те же, что в функции `prep` из проекта: температура, дождь, праздник, акция, день недели, месяц, выходной, номер дня `day_n`, кофейня и формат. Разница одна: кофейню и формат оставляем **текстом**, без `get_dummies`. Каждая библиотека получит категории в том виде, который понимает сама.

Для апрельского подбора и июльского выпуска строим два прогноза погоды из истории, доступной на каждую дату. После даты выпуска температуру и долю дождливых дней заменяем историческими средними для календарного месяца.

In [ ]:
s = pd.read_csv(DATA / "daily_sales.csv", parse_dates=["date"])
s = s.sort_values(["shop_id", "date"]).reset_index(drop=True)

def prep(s, issued):
    s = s.copy()
    history = s[s["date"] < issued]
    future = s["date"] >= issued
    for field in ("temp", "rain"):
        s[field] = s[field].astype(float)
        monthly = history.groupby(history["date"].dt.month)[field].mean()
        s.loc[future, field] = s.loc[future, "date"].dt.month.map(monthly)
    s["weekday"] = s["date"].dt.weekday
    s["month"] = s["date"].dt.month
    s["weekend"] = (s["weekday"] >= 5).astype(int)
    s["day_n"] = (s["date"] - pd.Timestamp("2024-01-01")).dt.days
    cols = ["temp", "rain", "holiday", "promo", "weekday", "month", "weekend", "day_n", "shop_id", "format"]
    return s[cols]

CAT = ["shop_id", "format"]
X_dev = prep(s, "2025-04-01")
X = prep(s, "2025-07-01")                                                     # категории — строки (для CatBoost)
X_cat = X.astype({c: "category" for c in CAT})                  # тип category (для LightGBM и HGB)
dev_cat = X_dev.astype({c: X_cat[c].dtype for c in CAT})
dev_dum = pd.get_dummies(X_dev, columns=CAT, dtype=int)
X_dum = pd.get_dummies(X, columns=CAT, dtype=int)               # флаги (классический вариант)
y = s["revenue_k"]

print("строк:", len(X), "| признаков:", X.shape[1], "| столбцов после get_dummies:", X_dum.shape[1])
X.head()

**Вывод по разделу 1.** Один набор признаков в трёх видах: строки, тип `category` и флаги. После `get_dummies` столбцов становится 17 вместо 10. На шести кофейнях это не страшно, но с тысячами товаров или адресов таблица флагов раздулась бы до тысяч почти пустых столбцов.

## 2. Разбиение по времени

Как в проекте: прогноз нужен на будущее, поэтому проверяем «прошлое → будущее».

- **обучение** (`fit`) — до 1 апреля 2025;
- **валидация** (`val`) — апрель–июнь 2025: по ней ранняя остановка выбирает число деревьев;
- **тест** (`test`) — июль–декабрь 2025: его модели не видят до финальной оценки.

Тест для ранней остановки брать нельзя. Иначе число деревьев подобрано по тесту, и оценка на нём получится оптимистичной.

In [ ]:
fit = s["date"] < "2025-04-01"
val = (s["date"] >= "2025-04-01") & (s["date"] < "2025-07-01")
test = s["date"] >= "2025-07-01"

for name, mask in [("fit", fit), ("val", val), ("test", test)]:
    print(f"{name:5} {mask.sum():5} строк, {s.loc[mask, 'date'].min():%Y-%m-%d} — {s.loc[mask, 'date'].max():%Y-%m-%d}")
refit = fit | val

**Вывод по разделу 2.** Три непересекающихся периода идут друг за другом. Все модели учатся на одном и том же `fit`, останавливаются по одному `val` и проверяются на одном `test`, поэтому сравнение честное.

## 3. Обучение и замер

Для каждой модели запишем строку таблицы: качество на тесте (RMSE и MAE, тыс. ₽), смещение (средняя ошибка «факт − прогноз»: больше нуля — модель занижает), сколько деревьев выбрала ранняя остановка, время обучения и время прогноза. Секундомер — `time.perf_counter()`, как в главе 9. Прогноз быстрый, поэтому его повторяем 5 раз и берём лучшее время.

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def timed(f, repeat=1):
    # Запускает f() repeat раз; возвращает результат и лучшее время в секундах.
    best = None
    for _ in range(repeat):
        t0 = time.perf_counter()
        result = f()
        t = time.perf_counter() - t0
        best = t if best is None else min(best, t)
    return result, best

rows = []

def add_row(name, model, X_test, n_trees, fit_s, search_s):
    pred, pred_s = timed(lambda: model.predict(X_test), repeat=5)
    rows.append({"модель": name, "RMSE": rmse(y[test], pred), "MAE": mean_absolute_error(y[test], pred),
                 "смещение": float((y[test] - pred).mean()),        # > 0: прогноз в среднем занижен
                 "деревьев": int(n_trees), "final fit, с": fit_s, "подбор и final fit, с": search_s, "predict, мс": pred_s * 1000})
    print(rows[-1])

### HistGradientBoosting из sklearn

Встроенная ранняя остановка HGB берёт валидацию случайной долей строк, а нам нужна валидация по времени. Поэтому делаем, как в шаге 6 проекта: обучаем 1000 деревьев, по `staged_predict` находим лучшее число деревьев на `val` и переобучаем модель с этим числом. Таблица показывает отдельно время подбора с переобучением и время последнего обучения на всех данных до июля.

In [ ]:
def hgb_by_val(dev, final, **extra):
    started = time.perf_counter()
    params = dict(learning_rate=0.05, max_iter=1000, early_stopping=False, random_state=RANDOM_STATE, **extra)
    probe = HistGradientBoostingRegressor(**params).fit(dev[fit], y[fit])
    curve = [rmse(y[val], p) for p in probe.staged_predict(dev[val])]
    params["max_iter"] = int(np.argmin(curve)) + 1
    model, fit_s = timed(lambda: HistGradientBoostingRegressor(**params).fit(final[refit], y[refit]))
    return model, params["max_iter"], fit_s, time.perf_counter()-started

m, k, t, total_s = hgb_by_val(dev_dum, X_dum)
add_row("HGB, get_dummies", m, X_dum[test], k, t, total_s)
m, k, t, total_s = hgb_by_val(dev_cat, X_cat, categorical_features="from_dtype")
add_row("HGB, категории", m, X_cat[test], k, t, total_s)

### LightGBM

Категории LightGBM понимает по типу `category`. Валидацию передают в `eval_X` и `eval_y` (так с версии 4.7; в старых версиях — `eval_set=[(X_val, y_val)]`). Ранняя остановка — обратный вызов `lgb.early_stopping(100)`: если 100 деревьев подряд не улучшили ошибку на валидации, обучение останавливается. Лучшее число деревьев хранится в `best_iteration_`, и `predict` использует именно его.

Обучим две версии. Первая — с настройками по умолчанию: до 31 листа в дереве и не меньше 20 строк в листе. Эти настройки рассчитаны на большие таблицы. Вторая — с мелкими деревьями: 7 листьев и 10 строк в листе.

In [ ]:
for name, leaves, min_rows in [("LightGBM", 31, 20), ("LightGBM, 7 листьев", 7, 10)]:
    started = time.perf_counter()
    lg = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.05, num_leaves=leaves, min_child_samples=min_rows,
                           random_state=RANDOM_STATE, verbose=-1, n_jobs=THREADS)
    _, fit_s = timed(lambda: lg.fit(dev_cat[fit], y[fit], eval_X=dev_cat[val], eval_y=y[val],
                                    callbacks=[lgb.early_stopping(100, verbose=False)]))
    chosen = lg.best_iteration_
    lg = lgb.LGBMRegressor(n_estimators=chosen, learning_rate=.05, num_leaves=leaves, min_child_samples=min_rows, random_state=RANDOM_STATE, verbose=-1,n_jobs=THREADS)
    _, final_fit = timed(lambda: lg.fit(X_cat[refit],y[refit]))
    add_row(name, lg, X_cat[test], chosen, final_fit, time.perf_counter()-started)

### CatBoost

CatBoost получает категории строками и список их имён в `cat_features`. Внутри он заменяет категорию средним ответом по ней, посчитанным только по «предыдущим» строкам: так ответ текущей строки не подсказывает сам себе. `early_stopping_rounds=100` и `use_best_model=True` — та же ранняя остановка по `eval_set`. `allow_writing_files=False` — не создавать рядом с блокнотом папку `catboost_info` с логами обучения.

In [ ]:
started = time.perf_counter()
cb = CatBoostRegressor(iterations=3000, learning_rate=0.05, depth=6, random_seed=RANDOM_STATE,
                       verbose=0, early_stopping_rounds=100, thread_count=THREADS, allow_writing_files=False)
_, fit_s = timed(lambda: cb.fit(X_dev[fit], y[fit], cat_features=CAT,
                                eval_set=(X_dev[val], y[val]), use_best_model=True))
chosen = cb.get_best_iteration()+1
cb = CatBoostRegressor(iterations=chosen,learning_rate=.05,depth=6,random_seed=RANDOM_STATE,verbose=0,thread_count=THREADS,allow_writing_files=False)
_, final_fit = timed(lambda: cb.fit(X[refit],y[refit],cat_features=CAT))
add_row("CatBoost",cb,X[test],chosen,final_fit,time.perf_counter()-started)

**Вывод по разделу 3.** Все пять моделей обучены на одних данных и остановлены по одной валидации. Число деревьев у них разное: у каждой библиотеки свой размер и своё устройство дерева. Поэтому сравнивать модели надо по качеству на тесте и по времени, а не по числу деревьев.

## 4. Сравнение

In [ ]:
table = pd.DataFrame(rows).set_index("модель").sort_values("RMSE")
table.round({"RMSE": 2, "MAE": 2, "смещение": 2, "fit, с": 2, "predict, мс": 1})

In [ ]:
imp = pd.Series(cb.get_feature_importance(), index=X.columns).sort_values(ascending=False)
imp.round(1)

**Прочитайте свою таблицу.** Сравните ошибки финального полугодия, число деревьев и смещение. Объясните, где бюджет ограничивает обучение, а где — число пакетных прогнозов. Время подбора включает раннюю остановку и финальное переобучение. Колонка final fit показывает только второй шаг. Итоги периода понадобятся для отчёта о качестве уже выбранных настроек.

## 5. Попробуйте сами

1. Поменяйте `learning_rate` на 0.1 и 0.02 у LightGBM и CatBoost. Как меняются число деревьев, время и RMSE?
2. Передайте в CatBoost таблицу `X_dum` без `cat_features`. Хуже ли стало качество?
3. Перенесите сюда приём из шага 7 проекта: обучите бустинги на `y / trend` без `day_n` и сравните RMSE.
4. Попробуйте у LightGBM `num_leaves=15` и `min_child_samples=5`. У LightGBM дерево растёт «по листьям», и его сложность задаёт `num_leaves`, а не глубина.